In [ ]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

# 기본적으로 현재 작업 디렉터리에 있는 .env 파일을 자동으로 찾아서 로드함.
load_dotenv()

In [ ]:
# 검색 도구 정의.
# max_results 를 통해 검색 결과를 최대 2개까지 반환하도록 설정. 그리고 랭그래프에 넘기기 위해 도구들을 리스트로 묶어줌.
tool = TavilySearch(max_results = 2)
tools = [tool]

#tool.invoke("랭그래프에서 노드란 무엇인가?")

# 실행결과는 다음과 같이 다양한 정보를 포함하는 딕셔너리 형태로 반환됨.
# .query : 실제로 검색한 질문
# .follow_up_questions : 추가 질문이 필요한 경우 해당 질문이 여기에 나타남.
# .answer : 도구가 직접 생성한 최종 답변, 
# .images : 관련 이미지가 있으면 여기에 포함됨
# .results
#   .url : 참고할 만한 웹 문서 링크.
#   .title : 해당 문서의 제목.
#   .content : 문서의 주요 요약이나 본문 일부
#   .score: 검색 결과의 신뢰도/유사도 점수(0~1 사이, 높을 수록 관련성이 높음.)
#   .raw_content : 원본 전체 내용(여기서는 제공되지 않음)
#   .response_time : 검색에 걸린 시간(초) 

In [ ]:
# 그래프에 도구 연동.
# 다음 단계에서는 StateGraph(상태 그래프)에 검색 도구를 연동하는 과정을 살펴봄.

# 우선 사용할 LLM 을 초기화 함. 그리고 bind_tools() 메서드로 해당 LLM이 어떤 도구들을 사용할 수 있는지 명시함.
# 이렇게 하면 LLM은 대화 도중 필요에 따라 검색 도구를 직접 호울할 수 있게 됨.
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4.1")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# 이제 챗봇 노드를 정의할 때 기존 LLM 대신 llm_with_tools 를 사용함.
# 다음 예시는 챗봇이 상태(State)내 메시지 이력을 입력받아 필요 시 도구를 호출해 최신 정보까지 반영한 답변을 생성하는 구조를 보여줌.
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 정의된 챗본 노드를 그래프에 추가함.
graph_builder.add_node("chatbot", chatbot)

# 이 과정을 통해 랭그래프내 챗봇은 단순 대화뿐만 아니라 실시간 웹 검색 결과까지 반영한 고도화된 답변을 생성할 수 있음.

In [ ]:
# 이제 도구 노드와 조건부 흐름을 추가해 챗봇이 도구 사용이 필요한지 여부에 따라 자동으로 검색 기능을 호출하고, 그 결과를 다시 대화에 반영할 수있도록 워크플로우를 확장해 보겠음.
# 도수 실행 노드 만들기.
# 챗봇이 도구를 호출할때 실제로 이 도구를 실행할 별도의 노드를 만들어야 함. 랭그래프는 이를 위해 두가지 방법을 제공함.

# 1. 직접 노드를 정의하는 방법.
import json
# ToolMessage 
# LLM이 호출한 도구(Tool)의 실행 결과를 다시 LLM에게 전달할 때 사용하는 메시지 객체임.
# 파이썬 함수나 API를 실행한 후, 그 결과 값을 다시 LLM을 보낼때 사용하는 규격임.
# 속성
#   . content : 도구가 실제 수행하고 반환한 결과 데이터(대개 문자열)
#   . tool_call_id: 어떤 도구 호출 요청에 대한 답변인지 매칭해 주는 고유 ID(LLM이 처음에 준 ID 그대로 사용)
from langchain_core.messages import ToolMessage

class BasicToolNode:
    """챗봇이 요청한 도구를 실행하는 도구입니다."""

    def __init__(self, tools: list):
        self.tools_by_name = {tool.name for tool in tools}

    def __call__(self, inputs: dict, outputs: list):
        messages = inputs.get("messages", [])
        if not messages:
            raise ValueError("입력된 상태에서 메시지를 찾을 수 없습니다.")
            outputs = []

        for tool_call in messages[-1].tool_calls:
            tool_result = self.tools_by_name[tool_call["name"]].invoke(tool_call["args"])

            outputs.append(
                ToolMessage(
                    content = json.dump(tool_result), #json.dump() : 파이썬 객체 (딕셔너리, 리스트 등)를 JSON 형식의 "문자열(String)"로 변환해 주는 함수.
                    name = tool_call["name"],
                    tool_call_id = tool_call["id"]
                )
            )

        return {"messages": outputs}

tool_node = BasicToolNode(tools=[tool])

# 2. 랭그래프가 미리 만들어둔 ToolNode 를 활용하는 방법.
#   직접 노드를 정의하는 방법은 호출 메시지를 직접 관리할 수 있는 장점이 있지만 코드가 길어지고 복잡해질 수 있음.
#   그래서 일반적으로 랭그래프가 제공하는 미리 만들어진 prebuilt 라이브러리인 ToolNode 를 활용하는 것이 쉽고 효율적임.

from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools=[tool])

# 이 방법은 내부적으로 도구 호출 처리과정을 모두 포함하고 잇어 직접 구현할 필요없이 간단히 사용할수 있다는 장점이 있음.
# 앞의 두가지 방법으로는 도구 노드를 만들었다면 다음 코드를 통해 그래프에 도구 노드를 추가할 수 있음.
graph_builder.add_node("tools", tool_node)

In [ ]:
# 조건부 엣지 정의하기.